# UC-ASSET-3 — Search Assets Across Collections

**Persona:** Scientist / Platform Auditor

**Goal:** Search assets by filter criteria (role, asset type) within a catalog,
follow cursor-based pagination to retrieve all pages, and demonstrate the
cross-catalog search endpoint that requires sysadmin credentials.

**Search API surface:**
- `POST /assets/catalogs/{catalog_id}/assets-search` — catalog-scoped search
- `POST /assets/assets-search` — cross-catalog search (sysadmin only)

**Request body schema (`SearchQuery`):**
```json
{
  "filters": [{"field": "asset_type", "op": "eq", "value": "RASTER"}],
  "collection_id": null,
  "limit": 50,
  "offset": 0
}
```

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`,
`DYNASTORE_SYSADMIN_TOKEN`, `CATALOG_ID`

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL        = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN           = os.environ.get("DYNASTORE_TOKEN", "")
SYSADMIN_TOKEN  = os.environ.get("DYNASTORE_SYSADMIN_TOKEN", "")
CATALOG_ID      = os.environ.get("CATALOG_ID", "demo-catalog")

PAGE_SIZE = 50

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)

sysadmin_headers = {
    "Authorization": f"Bearer {SYSADMIN_TOKEN}",
    "Content-Type": "application/json",
}
sysadmin_client = httpx.Client(base_url=BASE_URL, headers=sysadmin_headers, timeout=30.0)

print(f"Connected to {BASE_URL}  catalog={CATALOG_ID}")

In [ ]:
# Catalog-scoped search: RASTER assets
# SearchQuery uses a list of AssetFilter objects:
#   field: dotted path into asset fields (asset_id, uri, asset_type, metadata.key)
#   op: eq | ne | contains | gt | lt | gte | lte
#   value: filter value

search_payload = {
    "filters": [
        {"field": "asset_type", "op": "eq", "value": "RASTER"},
    ],
    "limit": PAGE_SIZE,
    "offset": 0,
}

r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/assets-search",
    content=json.dumps(search_payload),
)
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

result = r.json()
# result may be a list or a dict with 'assets' + pagination fields
if isinstance(result, list):
    assets_page = result
    total = None
else:
    assets_page = result.get("assets", result.get("items", []))
    total = result.get("total")

print(f"\nPage 1: {len(assets_page)} asset(s)  total={total}")
for a in assets_page[:5]:
    print(f"  asset_id={a.get('asset_id'):<30}  type={a.get('asset_type')}")

In [ ]:
# Pagination: iterate using offset until all pages exhausted
# The assets-search endpoint uses offset/limit pagination (not cursor tokens).
# Increment offset by limit until the returned page is shorter than the page size.

all_assets = list(assets_page)  # seed with page 1
offset = PAGE_SIZE

while len(assets_page) == PAGE_SIZE:
    search_payload["offset"] = offset
    r = client.post(
        f"/assets/catalogs/{CATALOG_ID}/assets-search",
        content=json.dumps(search_payload),
    )
    assert r.status_code == 200, f"Pagination failed at offset={offset}: {r.status_code}: {r.text}"
    page_result = r.json()
    assets_page = (
        page_result if isinstance(page_result, list)
        else page_result.get("assets", page_result.get("items", []))
    )
    all_assets.extend(assets_page)
    print(f"  offset={offset:5d}  page_size={len(assets_page):3d}  cumulative={len(all_assets)}")
    offset += PAGE_SIZE

print(f"\nTotal RASTER assets retrieved: {len(all_assets)}")
if total is not None:
    assert len(all_assets) == total or total == 0, (
        f"Expected {total} assets, got {len(all_assets)}"
    )

In [ ]:
# Cross-catalog search: POST /assets/assets-search (sysadmin only)
# No catalog_id in path — scans all catalogs the caller can access.

global_payload = {
    "filters": [
        {"field": "asset_type", "op": "eq", "value": "RASTER"},
    ],
    "limit": PAGE_SIZE,
    "offset": 0,
}

r = sysadmin_client.post(
    "/assets/assets-search",
    content=json.dumps(global_payload),
)
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200 for sysadmin global search, got {r.status_code}: {r.text}"

global_result = r.json()
global_assets = (
    global_result if isinstance(global_result, list)
    else global_result.get("assets", global_result.get("items", []))
)
global_total = global_result.get("total") if isinstance(global_result, dict) else None

print(f"\nGlobal search returned {len(global_assets)} assets on page 1  total={global_total}")

## Edge cases

### Global search with regular token → 403

The cross-catalog `/assets/assets-search` endpoint requires sysadmin scope.
A regular catalog-scoped token must receive `HTTP 403 Forbidden`.

In [ ]:
# Regular (non-sysadmin) token against global endpoint → 403
r = client.post(
    "/assets/assets-search",
    content=json.dumps(global_payload),
)
print(r.status_code, r.text[:300])
assert r.status_code in (401, 403), (
    f"Expected 401 or 403 for regular token on global search, got {r.status_code}: {r.text}"
)
print(f"{r.status_code} confirmed — global search denied for non-sysadmin token.")

In [ ]:
# metadata JSONB field search — performance note
#
# Filtering on nested metadata keys (e.g. metadata.sensor) uses a JSONB containment
# operator on the PostgreSQL `assets.metadata` column.
# There is currently NO GIN index on this column.
#
# For catalogs with > ~100k assets this becomes a sequential scan and will
# degrade query latency significantly.
#
# Recommended DDL (run as platform admin, per tenant schema):
#   CREATE INDEX CONCURRENTLY idx_assets_metadata_gin
#     ON {catalog_id}.assets USING gin(metadata jsonb_path_ops);
#
# Until this index is added, restrict metadata filters to small catalogs or
# combine them with asset_type / catalog_id filters to reduce scan size.

metadata_payload = {
    "filters": [
        {"field": "metadata.sensor", "op": "eq", "value": "OLI-2"},
    ],
    "limit": 10,
    "offset": 0,
}

r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/assets-search",
    content=json.dumps(metadata_payload),
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

meta_result = r.json()
meta_assets = (
    meta_result if isinstance(meta_result, list)
    else meta_result.get("assets", meta_result.get("items", []))
)
print(f"metadata.sensor=OLI-2 → {len(meta_assets)} asset(s)")
print("WARN: no GIN index on metadata column — add idx_assets_metadata_gin for production scale.")

In [ ]:
# Soft-deleted assets are excluded from search by default
#
# When an asset is soft-deleted (DELETE without force=true), the `deleted_at`
# column is set to a non-NULL timestamp.  The assets-search implementation adds
# WHERE deleted_at IS NULL to all queries, so soft-deleted assets are invisible
# to normal searches.
#
# To inspect soft-deleted assets an admin must query the DB directly:
#   SELECT * FROM {catalog_id}.assets WHERE deleted_at IS NOT NULL;
#
# Hard-deletion (force=true) removes the row entirely — not recoverable.

print("Soft-delete behaviour:")
print("  DELETE /assets/catalogs/{id}/assets/{aid}           → soft-delete (deleted_at set)")
print("  DELETE /assets/catalogs/{id}/assets/{aid}?force=true → hard-delete (row removed)")
print("  assets-search always filters WHERE deleted_at IS NULL (soft-deleted invisible)")

client.close()
sysadmin_client.close()